# WhisperWatch live validation

Compare four review rankings on the same 500 external LibriSpeech clips. The goal is to find out whether the frozen hybrid catches more high-error live Whisper transcripts than native confidence at a 20% review budget.

Open in Google Colab. Choose **Runtime → Change runtime type → T4 GPU** and connect to a hosted runtime. Run setup and inspect the preflight. Run the audio and transcription cell only after approval. Upload the experiment ZIP from this workspace when prompted. Its code and metadata are separate from your original boosted model results ZIP.

## 1. Set up and verify the frozen sample

The bundled manifest was selected uniformly without replacement, using seed 20261114 and sorted LibriSpeech IDs, from 5,559 entries in the two official LibriSpeech English test subsets. These are external to Common Voice 17. The preflight compares source, recording and filename identifiers and the speaker ID strings against the pinned Whispered English train and dev shards. The external dataset has a different speaker ID system, so an ID mismatch cannot establish that readers are different people. The sampling frame contains audiobook narration, and Whisper Large V3 pretraining overlap cannot be ruled out.

[OpenSLR LibriSpeech and CC BY 4.0 attribution](https://www.openslr.org/12/) · [Pinned LibriSpeech metadata](https://huggingface.co/datasets/openslr/librispeech_asr/tree/8edc81a625bfc3d004c8a5d3870e0ade8e1cc59c) · [Whispered dataset card](https://huggingface.co/datasets/burakaydinofficial/Whispered/blob/25299fd560dde4271f5177118354af5a961b92fd/README.md)

In [ ]:
%pip install -q "pandas==2.2.3" "pyarrow>=23.0.1" "scikit-learn==1.8.0" "joblib==1.5.3"

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import json
import sys

from google.colab import files
from IPython.display import Markdown, display

project = Path("/content/whisperwatch_live_validation")
project.mkdir(exist_ok=True)

def upload_one(prompt):
    print(prompt)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Select one ZIP.")
    return BytesIO(next(iter(uploaded.values())))

with ZipFile(upload_one("Upload whisperwatch-live-validation-code.zip.")) as bundle:
    for member in bundle.namelist():
        parent, name = member.split("/")
        if parent == "src" and name.endswith(".py") or member in {"preflight/manifest.csv", "preflight/preflight.json"}:
            target = project / parent / name
            target.parent.mkdir(exist_ok=True)
            target.write_bytes(bundle.read(member))

sys.path.insert(0, str(project))
plan = project / "preflight"
print(json.dumps(json.loads((plan / "preflight.json").read_text()), indent=2))


## 2. Freeze the live decoder

Use the existing app transcription path with the same pinned Systran Large V3 weights as the 80-clip run (revision edaa852ec7e145841d8ffdb056a99866b5f0a478). The run uses CUDA float16, faster-whisper 1.2.1, CTranslate2 4.9.0, and PyAV 16.0.1. It forces English transcription and uses beam size 5, temperature fallback 0.0–1.0 in 0.2 steps, no initial prompt, VAD off, word timestamps off, and faster-whisper mono 16 kHz decoding. Export all effective options.

The archive documents Large V3, faster-whisper, float16 and pipeline 0.2.0, but omits the exact converted checkpoint, library versions, temperatures, beam/scoring settings, VAD and preprocessing. On the same audio in the prior pilot, every clip had one segment, so average and minimum log probability were the same. The production extractor uses an unweighted mean and minimum of finite segment scores. End timestamps moved later on 76/80, typical absolute change 0.58 s, even though full audio duration stayed fixed. Different timestamp/token paths, decoding settings, and conversion or preprocessing can change scores even for identical text. Missing archive settings prevent causal attribution. The new evaluation uses one configuration throughout.

**Approval checkpoint.** Stop here until the sample and resource estimate are approved. The next steps install GPU dependencies, fetch 683 MB of LibriSpeech shards and about 3.1 GB of Whisper weights if uncached, and run the 500 transcriptions.

In [ ]:
APPROVED = False  # Change to True after approval.
if not APPROVED:
    raise RuntimeError("Pause here for approval before installing GPU packages or downloading audio.")
%pip install -q "faster-whisper==1.2.1" "ctranslate2==4.9.0" "av==16.0.1" "huggingface-hub==1.32.0" "nvidia-cublas-cu12" "nvidia-cudnn-cu12==9.*"


## 3. Run fresh transcription and compare rankings

Upload the original whisperwatch-boosted-results.zip. The code verifies the original model hash, extracts the same ten features from the live output, and scores them without fitting any components. Native risk is negative average segment log probability (missing confidence gets highest native priority). The hybrid averages the two percentile ranks using average ranks for ties. Every method selects ceil(20% × evaluated clips), breaking ties by ascending clip ID. Random review uses a fixed seed. Human references only enter after ranking to calculate WER and labels. Normalization folds case and removes punctuation and symbols, without expanding numbers. Empty predictions with valid references are evaluable. Failures and unsupported durations stay in the status log and all methods use the same successful clips.

The notebook reports a paired reader bootstrap using 2,000 fixed-seed draws. Each draw resamples readers with replacement, ranks the same drawn recordings for every method, and reapplies the budget. Differences are reported against native confidence. A lead around 3–5 recall percentage points is encouraging, with the interval and the number of high-error positives determining confidence in the decision.

In [ ]:
if not APPROVED:
    raise RuntimeError("Obtain approval before launching the audio and GPU workload.")

from src.live_validation import run_approved

output = project / "results"
bundle = upload_one("Upload the original whisperwatch-boosted-results.zip.")
metrics, intervals, run_info = run_approved(plan, bundle, output)
display(Markdown((output / "report.md").read_text()))
display(metrics, intervals)


## 4. Download experiment results

The ZIP includes status logs, raw live segment output, all four rankings, error examples, the feature distribution table, bootstrap intervals, and the report. This cell can also save partial outputs after a failure. Audio, model weights, and Parquet shards stay out of the export.

In [ ]:
output = project / "results"
result_zip = Path("/content/whisperwatch-live-validation-results.zip")
from zipfile import ZIP_DEFLATED

with ZipFile(result_zip, "w", ZIP_DEFLATED) as bundle:
    for path in sorted(output.iterdir()):
        if path.suffix in {".csv", ".json", ".md"} and path.name != "live_input.json":
            bundle.write(path, path.name)
files.download(str(result_zip))
